In [1]:
import time
import re
import os
import random
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoAlertPresentException,
    UnexpectedAlertPresentException,
    NoSuchElementException,
)
from tqdm import tqdm


# =========================
# 🔧 0. 경로 & 기본 설정
# =========================

# 👉 전체 URL 크롤링용 엑셀 경로
EXCEL_PATH = "yeoshinticket_urls_part_1.xlsx"  # URL 목록 엑셀

# ✅ 출력 파일
OUTPUT_CSV = "yeoshinticket_detail_procedure_part_all_1.csv"
ERROR_LOG_CSV = "yeoshinticket_detail_procedure_errors_all.csv"

# ✅ 재시도 / 저장 설정
MAX_ROWS = None        # 🔹 전체 URL 모두 크롤링
SAVE_INTERVAL = 10     # N건마다 중간 저장
MAX_RETRY = 3         # 페이지 재시도 횟수

# =========================
# 🔍 테스트 크롤링 옵션
# =========================
TEST_MODE = False          # 🔹 테스트 모드 OFF → 전체 크롤링
TEST_SAMPLE_SIZE = 100     # (사용 안 됨, 값만 남겨둠)

BASE_URL = "https://www.yeoshin.co.kr"

# =========================
# 1. 기본 셀렉터
# =========================

# 시술명
SEL_TREATMENT_NAME = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.flex-col.gap-\\[4px\\] > h1"
)

# 메인 이미지
SEL_MAIN_IMAGE = (
    "#ct-view > div.relative.w-full > div.relative.overflow-hidden > div > img"
)

# 평점 / 후기수
SEL_RATING = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.flex-col.gap-\\[4px\\] > button > div > span"
)

SEL_REVIEW_COUNT = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.flex-col.gap-\\[4px\\] > button > span"
)

# 가격 정보 (정상가)
SEL_ORIGINAL_PRICE = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > div:nth-child(1) > div"
)

# 판매가 (할인 있는 경우: div > div > h2)
SEL_SELLING_PRICE_DISCOUNT = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > div > h2"
)

# 판매가 (할인 없는 경우: div > h2)
SEL_SELLING_PRICE_NO_DISCOUNT = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > h2"
)

# 옵션 설명
SEL_OPTION_INFO = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > div.flex.items-end.gap-\\[4px\\] > "
    "div.text-\\[12px\\].mb-\\[2px\\].font-normal.leading-\\[18px\\].text-gray500"
)

# 시술 해시태그(상단 작은 태그들)
SEL_TREATMENT_HASHTAGS = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.flex-col.gap-\\[4px\\] > div"
)

# VAT / 부가 정보
SEL_VAT_INFO = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > div.flex.items-end.gap-\\[4px\\] > "
    "div.text-\\[12px\\].mb-\\[2px\\].font-normal.leading-\\[18px\\].text-gray500"
)

# 병원명
SEL_HOSPITAL_NAME = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
    "article > div > div > h2"
)

# 병원 주소 – 옛날 방식(섹션 인덱스 기반, fallback용)
SEL_HOSPITAL_ADDRESS_OLD = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
    "article > section:nth-child(3) > div > div > div"
)

# 병원 정보(병원 한줄 설명, 특징 등)
SEL_HOSPITAL_INFO = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
    "article > section.flex.flex-wrap.gap-\\[4px\\].px-\\[16px\\]"
)

# 마취 정보 등 (하단 병원 섹션의 텍스트 블록)
SEL_ANESTHESIA_INFO = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
    "article > section.flex.flex-col.gap-\\[16px\\].px-\\[16px\\] > div > div"
)

# 시술 설명 (상단 설명 블록 – fallback용)
SEL_TREATMENT_DESCRIPTION = (
    "#ct-view > div.relative.w-full > div:nth-child(5) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > "
    "article:nth-child(1) > section"
)

# 시술시간/마취여부/회복시간/시술효과 묶음
SEL_DOWNTIME_INFO = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "section.flex.flex-col.px-\\[16px\\].py-\\[24px\\] > div:nth-child(1)"
)

# ====== 타겟/특장점/Q&A/주의사항 기본 fallback 셀렉터 ======

# 시술 특장점 (fallback)
SEL_PROCEDURE_INFO_FALLBACK = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > article.relative > section"
)

# 추천대상 (fallback)
SEL_TARGET_GROUP_FALLBACK = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > article:nth-child(32) > section"
)

# Q&A (fallback)
SEL_PROCEDURE_QNA_FALLBACK = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > "
    "article.pt-\\[32px\\].pb-\\[40px\\].px-\\[24px\\].flex.flex-col.gap-\\[32px\\]"
)

# 주의사항 (= 부작용 안내)
SEL_SIDE_EFFECT_INFO = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > article.p-\\[12px\\] > section"
)


# =========================
# 2. 드라이버 / 유틸 함수
# =========================

def init_driver():
    """크롬 드라이버 초기화 (headless + 이미지 로딩 끔)"""
    chromedriver_autoinstaller.install()
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")

    # 이미지 로딩 끄기(속도 개선)
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)
    return driver


def safe_text(driver, selector):
    if not selector:
        return None
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        return el.text.strip()
    except Exception:
        return None


def safe_attr(driver, selector, attr):
    if not selector:
        return None
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        val = el.get_attribute(attr)
        return val.strip() if val else None
    except Exception:
        return None


def clean_lines(raw_text):
    """여러 줄 텍스트를 라인별로 정리 + 공백 정리 + 중복 제거"""
    if not raw_text:
        return []

    lines = []
    for ln in raw_text.splitlines():
        txt = re.sub(r"\s+", " ", ln).strip()
        if not txt:
            continue
        lines.append(txt)

    if not lines:
        return []

    seen = set()
    uniq = []
    for txt in lines:
        if txt in seen:
            continue
        seen.add(txt)
        uniq.append(txt)
    return uniq


def safe_section_text(driver, selector, sep=" "):
    """selector로 잡힌 영역 전체 텍스트를 정리해서 하나의 문자열로 리턴"""
    if not selector:
        return None

    try:
        root = driver.find_element(By.CSS_SELECTOR, selector)
        raw = root.text
    except Exception:
        return None

    uniq_lines = clean_lines(raw)
    return sep.join(uniq_lines) if uniq_lines else None


def section_text_from_element(element, sep=" "):
    """이미 찾은 element에서 텍스트 정리"""
    try:
        raw = element.text
    except Exception:
        return None

    uniq_lines = clean_lines(raw)
    return sep.join(uniq_lines) if uniq_lines else None


def extract_number(text):
    """문자열에서 숫자만 뽑아서 int로 변환 (없으면 None)"""
    if not text:
        return None
    digits = re.sub(r"[^\d]", "", text)
    return int(digits) if digits else None


def normalize_price_to_int(text):
    """
    가격 문자열에서 숫자만 뽑아서 int로 변환.
    예) '혜택가 224,900원' -> 224900
    """
    return extract_number(text)


# ========= 엑셀 저장 유틸 =========

def clean_dataframe_for_excel(df_out: pd.DataFrame) -> pd.DataFrame:
    """
    엑셀 저장용 컨트롤 문자 제거 + 너무 긴 문자열 잘라내기
    (엑셀 셀 최대 32767자 → 여유 있게 32000자로 제한)
    """
    illegal_pattern = re.compile(r"[\x00-\x08\x0b-\x0c\x0e-\x1f]")
    MAX_LEN = 32000

    def clean_for_excel(val):
        if pd.isna(val):
            return val
        s = str(val)
        s = illegal_pattern.sub("", s)
        if len(s) > MAX_LEN:
            s = s[:MAX_LEN]
        return s

    obj_cols = df_out.select_dtypes(include=["object"]).columns
    for col in obj_cols:
        df_out[col] = df_out[col].apply(clean_for_excel)
    return df_out


def save_results(results, path, final=False):
    """중간/최종 저장 (항상 엑셀 안전화 + 시트 이름 고정)"""
    if not results:
        if final:
            print("⚠️ 크롤링된 데이터가 없어 빈 CSV/XLSX 파일만 생성했습니다.")
        return

    df_out = pd.DataFrame(results)
    df_out = clean_dataframe_for_excel(df_out)

    # CSV
    df_out.to_csv(path, index=False, encoding="utf-8-sig")

    # XLSX (시트 이름 고정: data)
    xlsx_path = os.path.splitext(path)[0] + ".xlsx"
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            df_out.to_excel(writer, index=False, sheet_name="data")
    except Exception as e:
        print(f"⚠️ XLSX 저장 중 에러 발생: {e}")

    if final:
        print(f"[최종 저장] {len(df_out)}건 → {path}, {xlsx_path}")
    else:
        print(f"[중간 저장] {len(df_out)}건 → {path}, {xlsx_path}")


def log_error_row(url, depth1, depth2, depth3, exc):
    err_info = {
        "event_url": url,
        "대분류": depth1,
        "중분류": depth2,
        "소분류": depth3,
        "error_type": type(exc).__name__,
        "error_msg": str(exc),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    df_err = pd.DataFrame([err_info])

    # CSV append
    if os.path.exists(ERROR_LOG_CSV):
        df_err.to_csv(
            ERROR_LOG_CSV,
            mode="a",
            header=False,
            index=False,
            encoding="utf-8-sig",
        )
    else:
        df_err.to_csv(ERROR_LOG_CSV, index=False, encoding="utf-8-sig")

    # 에러 로그 xlsx 동기화 (실패해도 조용히 패스)
    try:
        df_all = pd.read_csv(ERROR_LOG_CSV)
        df_all = clean_dataframe_for_excel(df_all)
        err_xlsx = os.path.splitext(ERROR_LOG_CSV)[0] + ".xlsx"
        with pd.ExcelWriter(err_xlsx, engine="openpyxl") as writer:
            df_all.to_excel(writer, index=False, sheet_name="error_log")
    except Exception as e:
        print(f"⚠️ 에러 로그 XLSX 동기화 중 에러 발생: {e}")


# =========================
# 3. 병원 주소 / 추천대상 / 특장점 / Q&A / 가격 헬퍼
# =========================

def get_hospital_address(driver):
    """
    주소복사 버튼 기준으로 주소 추출.
    1순위: '주소복사' 버튼 바로 앞 span
    2순위: 기존 CSS 셀렉터(섹션 인덱스 기반)
    """
    try:
        copy_btn = driver.find_element(
            By.XPATH,
            "//button[.//span[contains(normalize-space(.), '주소복사')]]"
        )
        addr_span = copy_btn.find_element(
            By.XPATH,
            "./preceding-sibling::span[1]"
        )
        addr = addr_span.text.strip()
        if addr:
            return addr
    except NoSuchElementException:
        pass
    except Exception:
        pass

    # fallback
    return safe_text(driver, SEL_HOSPITAL_ADDRESS_OLD)


def get_treatment_description(driver):
    """
    시술 설명(treatment_description)을 최대한 안정적으로 가져오기.
    1순위: background-color 있는 article + 본문 p 포함 (설명 박스)
    2순위: 기존 CSS 셀렉터(SEL_TREATMENT_DESCRIPTION)
    """
    try:
        article = driver.find_element(
            By.XPATH,
            "//article[contains(@style,'background-color') and "
            ".//p[contains(@class,'leading-')]]"
        )
        text = section_text_from_element(article, sep=" ")
        if text:
            return text
    except Exception:
        pass

    # fallback
    return safe_section_text(driver, SEL_TREATMENT_DESCRIPTION, sep=" ")


def get_target_group(driver):
    """
    추천대상:
    - (신규 레이아웃) 'target' 일러스트 이미지 + 중앙 정렬 텍스트(p.text-center.break-keep)
    - (구 레이아웃) '추천'과 '대상'이 함께 들어간 h2/h3 아래의 중앙 정렬 텍스트
    - (추가) '추천'만 포함된 h2/h3 + 중앙 정렬 텍스트
    - 최종: 구형 CSS fallback
    """
    # 0순위: 일러스트 이미지(src에 'target' 포함) + 중앙정렬 텍스트
    try:
        article = driver.find_element(
            By.XPATH,
            "//article["
            "  .//img[contains(@src,'target')]"
            "  and .//p[contains(@class,'text-center') and contains(@class,'break-keep')]"
            "]"
        )
        ps = article.find_elements(
            By.XPATH,
            ".//p[contains(@class,'text-center') and contains(@class,'break-keep')]"
        )
        texts = [p.text.strip() for p in ps if p.text.strip()]
        if texts:
            uniq = []
            seen = set()
            for t in texts:
                if t in seen:
                    continue
                seen.add(t)
                uniq.append(t)
            return " / ".join(uniq)
    except Exception:
        pass

    # 기존 1순위: '추천' + '대상'이 같이 들어간 헤더
    try:
        article = driver.find_element(
            By.XPATH,
            "//article["
            "  .//h2[(contains(normalize-space(.), '추천') and contains(normalize-space(.), '대상'))]"
            "   or "
            "  .//h3[(contains(normalize-space(.), '추천') and contains(normalize-space(.), '대상'))]"
            "]"
        )
        ps = article.find_elements(
            By.XPATH,
            ".//p[contains(@class,'text-center') and contains(@class,'break-keep')]"
        )
        texts = [p.text.strip() for p in ps if p.text.strip()]
        if texts:
            uniq = []
            seen = set()
            for t in texts:
                if t in seen:
                    continue
                seen.add(t)
                uniq.append(t)
            return " / ".join(uniq) if uniq else None
    except Exception:
        pass

    # 2순위: '추천'만 들어간 헤더 (이런 분께 추천해요 등)
    try:
        article = driver.find_element(
            By.XPATH,
            "//article["
            "  (.//h2[contains(normalize-space(.), '추천')]"
            "   or .//h3[contains(normalize-space(.), '추천')])"
            "  and .//p[contains(@class,'text-center') and contains(@class,'break-keep')]"
            "]"
        )
        ps = article.find_elements(
            By.XPATH,
            ".//p[contains(@class,'text-center') and contains(@class,'break-keep')]"
        )
        texts = [p.text.strip() for p in ps if p.text.strip()]
        if texts:
            uniq = []
            seen = set()
            for t in texts:
                if t in seen:
                    continue
                seen.add(t)
                uniq.append(t)
            return " / ".join(uniq) if uniq else None
    except Exception:
        pass

    # 최종 fallback: 옛날 전체 섹션 텍스트
    return safe_section_text(driver, SEL_TARGET_GROUP_FALLBACK, sep=" ")


def get_procedure_info(driver):
    """
    시술 특장점:
    1순위: h2/h3 안에 '시술'과 '특장점'이 함께 들어간 article 전체 텍스트
    2순위: CSS fallback
    """
    try:
        article = driver.find_element(
            By.XPATH,
            "//article["
            "  .//h2[(contains(normalize-space(.), '시술') and contains(normalize-space(.), '특장점'))]"
            "   or "
            "  .//h3[(contains(normalize-space(.), '시술') and contains(normalize-space(.), '특장점'))]"
            "]"
        )
        text = section_text_from_element(article, sep=" ")
        if text:
            return text
    except Exception:
        pass

    return safe_section_text(driver, SEL_PROCEDURE_INFO_FALLBACK, sep=" ")


def get_procedure_qna(driver):
    """
    자주 묻는 질문(Q&A):
    - '자주'와 '질문'이 함께 들어간 h2/h3를 가진 article 전체 텍스트.
    """
    try:
        article = driver.find_element(
            By.XPATH,
            "//article["
            "  .//h2[(contains(normalize-space(.), '자주') and contains(normalize-space(.), '질문'))]"
            "   or "
            "  .//h3[(contains(normalize-space(.), '자주') and contains(normalize-space(.), '질문'))]"
            "]"
        )
        return section_text_from_element(article, sep=" ")
    except Exception:
        pass

    return safe_section_text(driver, SEL_PROCEDURE_QNA_FALLBACK, sep=" ")


def get_side_effect_info(driver):
    return safe_section_text(driver, SEL_SIDE_EFFECT_INFO, sep=" ")


def get_price_fields(driver):
    """
    original_price / selling_price를 모두 int로 반환.
    - original_price: 정상가(있으면)
    - selling_price: 실제 판매가 (할인 여부 관계없이 h2)
    """
    original_raw = safe_text(driver, SEL_ORIGINAL_PRICE)

    # 1순위: 할인 있는 구조 div > div > h2
    selling_raw = safe_text(driver, SEL_SELLING_PRICE_DISCOUNT)

    # 2순위: 할인 없는 구조 div > h2
    if not selling_raw:
        selling_raw = safe_text(driver, SEL_SELLING_PRICE_NO_DISCOUNT)

    original_price = normalize_price_to_int(original_raw)
    selling_price = normalize_price_to_int(selling_raw)

    return original_price, selling_price


def get_weekly_opening_hours(driver):
    """영업시간 토글 열고 일주일 영업시간 텍스트 가져오기"""
    css_hours_button = "#radix-\\:Rl3sckum\\:"
    css_hours_panel = "#radix-\\:R1l3sckum\\:"

    try:
        btn = driver.find_element(By.CSS_SELECTOR, css_hours_button)
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.4)
    except Exception:
        pass

    try:
        panel = driver.find_element(By.CSS_SELECTOR, css_hours_panel)
        raw_text = panel.text
        lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]
        if not lines:
            return None
        return " / ".join(lines)
    except Exception:
        return None


def get_hospital_contact(driver):
    """
    병원 연락처 추출 (class 기반, nth-child 안 씀)
    """
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        "div.text-gray700.font-normal.text-\\[12px\\].leading-\\[18px\\]"
    )

    texts = []
    for c in containers:
        try:
            for ln in c.text.splitlines():
                ln = ln.strip()
                if ln:
                    texts.append(ln)
        except Exception:
            continue

    if not texts:
        return None

    phone_pattern = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")

    # 1순위: 전화번호 패턴이 들어있는 줄
    for t in texts:
        if phone_pattern.search(t):
            return t

    # 2순위: 숫자가 가장 많이 들어있는 줄
    texts_sorted = sorted(texts, key=lambda x: sum(ch.isdigit() for ch in x), reverse=True)
    best = texts_sorted[0]
    if sum(ch.isdigit() for ch in best) < 4:
        return None
    return best


# =========================
# 4. 단일 페이지 크롤링
# =========================

def crawl_event_page(driver, url, depth1=None, depth2=None, depth3=None):
    driver.get(url)

    # 메인 컨테이너 로딩 대기
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "#ct-view"))
    )

    # 시술명
    treatment_name = safe_text(driver, SEL_TREATMENT_NAME)

    # 메인 이미지
    main_image_url = safe_attr(driver, SEL_MAIN_IMAGE, "src")

    # 시술 해시태그
    treatment_hashtags = safe_text(driver, SEL_TREATMENT_HASHTAGS)

    # 평점
    rating_text = safe_text(driver, SEL_RATING)
    try:
        rating = float(rating_text) if rating_text else None
    except ValueError:
        rating = None

    # 후기 수 (숫자만, '개' 제거 → int)
    review_count_raw = safe_text(driver, SEL_REVIEW_COUNT)
    review_count = extract_number(review_count_raw)

    # 가격 정보 (정상가 / 판매가 → int)
    original_price, selling_price = get_price_fields(driver)

    # VAT / 부가 정보
    vat_info = safe_text(driver, SEL_VAT_INFO)

    # 옵션 정보
    option_info = safe_text(driver, SEL_OPTION_INFO)

    # 병원 정보
    hospital_name = safe_text(driver, SEL_HOSPITAL_NAME)
    hospital_address = get_hospital_address(driver)
    hospital_info = safe_section_text(driver, SEL_HOSPITAL_INFO, sep=" ")

    # 시술 설명
    treatment_description = get_treatment_description(driver)

    # 시술 특장점
    procedure_info = get_procedure_info(driver)

    # 추천대상
    target_group = get_target_group(driver)

    # Q&A
    procedure_qna = get_procedure_qna(driver)

    # 다운타임 / 시술시간 / 마취여부 / 시술효과
    downtime_info = safe_section_text(driver, SEL_DOWNTIME_INFO, sep=" ")

    # 주의사항 (부작용 안내)
    side_effect_info = get_side_effect_info(driver)

    # 마취 정보
    anesthesia_info = safe_section_text(driver, SEL_ANESTHESIA_INFO, sep=" ")

    # 영업시간
    opening_hours = get_weekly_opening_hours(driver)

    # 병원 연락처 (안심번호)
    hospital_contact = get_hospital_contact(driver)

    row = {
        "event_url": url,
        "대분류": depth1,
        "중분류": depth2,
        "소분류": depth3,
        "treatment_name": treatment_name,
        "treatment_hashtags": treatment_hashtags,
        "main_image_url": main_image_url,
        "rating": rating,
        "review_count": review_count,        # 🔢 int
        "original_price": original_price,    # 🔢 int
        "selling_price": selling_price,      # 🔢 int
        "vat_info": vat_info,
        "hospital_name": hospital_name,
        "hospital_address": hospital_address,
        "hospital_info": hospital_info,
        "treatment_description": treatment_description,
        "option_info": option_info,
        "target_group": target_group,
        "procedure_info": procedure_info,
        "procedure_qna": procedure_qna,
        "downtime_info": downtime_info,
        "side_effect_info": side_effect_info,
        "anesthesia_info": anesthesia_info,
        "opening_hours": opening_hours,
        "병원_연락처(안심번호)": hospital_contact,
    }
    return row


def crawl_event_page_with_retry(driver, url, depth1=None, depth2=None, depth3=None):
    last_exc = None
    for attempt in range(1, MAX_RETRY + 1):
        try:
            return crawl_event_page(driver, url, depth1, depth2, depth3)
        except (TimeoutException, UnexpectedAlertPresentException) as e:
            last_exc = e
            try:
                alert = driver.switch_to.alert
                alert.dismiss()
            except NoAlertPresentException:
                pass
            time.sleep(2)
        except Exception as e:
            last_exc = e
            break
    raise last_exc


# =========================
# 5. 메인 실행부 (단일 드라이버, 순차 크롤링)
# =========================

def main():
    if not os.path.exists(EXCEL_PATH):
        raise FileNotFoundError(f"{EXCEL_PATH} 파일을 찾을 수 없습니다.")

    df = pd.read_excel(EXCEL_PATH)

    if "event_url" not in df.columns:
        raise ValueError("엑셀에 'event_url' 컬럼이 없습니다. 컬럼명을 확인해주세요.")

    total_rows = len(df)
    print(f"[정보] 전체 URL 수: {total_rows}개")

    # 테스트 모드 / MAX_ROWS 적용
    if TEST_MODE:
        sample_n = min(TEST_SAMPLE_SIZE, total_rows)
        df_target = df.sample(n=sample_n, random_state=42).reset_index(drop=True)
        print(f"[테스트 모드] 랜덤 {sample_n}개 URL만 크롤링합니다.")
    else:
        if MAX_ROWS is not None:
            df_target = df.head(MAX_ROWS).reset_index(drop=True)
            print(f"[부분 실행] 상위 {len(df_target)}개 URL만 크롤링합니다.")
        else:
            df_target = df.reset_index(drop=True)
            print(f"[전체 실행] 전체 {len(df_target)}개 URL 크롤링합니다.")

    if len(df_target) == 0:
        print("⚠️ 크롤링할 URL이 없습니다.")
        return

    results = []

    # 재시작용: 기존 결과 불러오기
    if os.path.exists(OUTPUT_CSV):
        try:
            df_done = pd.read_csv(OUTPUT_CSV)
            if "event_url" in df_done.columns:
                done_urls = set(df_done["event_url"].dropna().astype(str))
                # 이번 타겟 중 이미 수집된 것 제외
                df_target = df_target[~df_target["event_url"].astype(str).isin(done_urls)]
                results.extend(df_done.to_dict("records"))
                print(
                    f"[재시작] 기존 수집 건수: {len(results)}개 "
                    f"(이번에 새로 수집할 URL: {len(df_target)}개)"
                )
        except Exception as e:
            print(f"[경고] 기존 결과 파일 읽기 실패. 새로 시작합니다. 에러: {e}")

    total_to_crawl = len(df_target)
    if total_to_crawl == 0:
        print("✅ 모든 대상 URL이 이미 수집되었습니다.")
        save_results(results, OUTPUT_CSV, final=True)
        return

    driver = init_driver()
    print(f"[실행 정보] 실제 크롤링할 URL 수: {total_to_crawl}개")

    try:
        # tqdm 진행률 바 적용
        for idx, row in enumerate(
            tqdm(df_target.itertuples(index=False), total=total_to_crawl, desc="크롤링 진행"),
            start=1
        ):
            url = getattr(row, "event_url")
            depth1 = getattr(row, "대분류", None) if "대분류" in df_target.columns else None
            depth2 = getattr(row, "중분류", None) if "중분류" in df_target.columns else None
            depth3 = getattr(row, "소분류", None) if "소분류" in df_target.columns else None

            url = str(url)

            print(f"[{idx}/{total_to_crawl}] 크롤링 중 → {url}")

            try:
                detail = crawl_event_page_with_retry(
                    driver, url, depth1=depth1, depth2=depth2, depth3=depth3
                )
                results.append(detail)

            except Exception as e:
                print(f"⚠️ URL 스킵: {url} ({type(e).__name__}: {e})")
                log_error_row(url, depth1, depth2, depth3, e)

            # 중간 저장
            if idx % SAVE_INTERVAL == 0:
                save_results(results, OUTPUT_CSV, final=False)

            # 서버/클라이언트 부담 줄이려고 살짝 딜레이
            time.sleep(1.0)

    finally:
        driver.quit()

    # 최종 저장
    save_results(results, OUTPUT_CSV, final=True)
    print(
        f"\n✅ 크롤링 종료. 최종 {len(results)}건 저장 → {OUTPUT_CSV}"
    )
    print("✅ CSV / XLSX 저장 완료")


if __name__ == "__main__":
    try:
        main()
    except Exception as main_e:
        print(f"\n❌ 프로그램 메인 실행 중 치명적인 에러 발생: {main_e}")


[정보] 전체 URL 수: 688개
[전체 실행] 전체 688개 URL 크롤링합니다.
[실행 정보] 실제 크롤링할 URL 수: 688개


크롤링 진행:   0%|          | 0/688 [00:00<?, ?it/s]

[1/688] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/16693


크롤링 진행:   0%|          | 1/688 [00:02<31:13,  2.73s/it]

[2/688] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/25183


...

[687/688] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/26566


크롤링 진행: 100%|█████████▉| 687/688 [26:53<00:02,  2.16s/it]

[688/688] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/23483


크롤링 진행: 100%|██████████| 688/688 [26:55<00:00,  2.35s/it]


[최종 저장] 688건 → yeoshinticket_detail_procedure_part_all_1.csv, yeoshinticket_detail_procedure_part_all_1.xlsx

✅ 크롤링 종료. 최종 688건 저장 → yeoshinticket_detail_procedure_part_all_1.csv
✅ CSV / XLSX 저장 완료
